In [1]:
import numpy as np
import pandas as pd
import src.csv_io as csv_io 
from pathlib import Path
from src.config import RAW_DIR

Question 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [2]:
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders.sort_values(["customer_id", "order_date", "order_id"])

inter_order_gap_days = (
    orders.groupby("customer_id")["order_date"]
    .diff()
    .dt.days
    .dropna()
)

q1_median_gap_days = float(np.median(inter_order_gap_days.to_numpy()))
print(f"Q1 answer: median inter-order gap ~ {q1_median_gap_days:.0f} days")


Q1 answer: median inter-order gap ~ 144 days


Question 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [3]:
products = csv_io.read_csv_file(RAW_DIR / "products.csv")
products = products.copy()

products["gross_margin_rate"] = (products["price"] - products["cogs"]) / products["price"]
segment_margin = (
    products.groupby("segment", as_index=False)["gross_margin_rate"]
    .mean()
    .sort_values("gross_margin_rate", ascending=False)
)

q2_best_segment = segment_margin.iloc[0]["segment"]
q2_best_margin = float(segment_margin.iloc[0]["gross_margin_rate"])

print(f"Q2 answer: best segment = {q2_best_segment} (avg gross margin ~ {q2_best_margin:.2%})")
segment_margin


Q2 answer: best segment = Standard (avg gross margin ~ 31.34%)


,segment,gross_margin_rate
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


Question 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?


In [4]:
products = csv_io.read_csv_file(RAW_DIR / "products.csv")
returns = csv_io.read_csv_file(RAW_DIR / "returns.csv")

products_cp = products.copy()
returns_cp = returns.copy()

products_cp["product_id"] = products_cp["product_id"].astype(str)
returns_cp["product_id"] = returns_cp["product_id"].astype(str)

merged = pd.merge(returns_cp, products_cp, how="left", on="product_id")

streetwear_returns = merged[merged["category"] == "Streetwear"]

q3_most_returned_reason = streetwear_returns["return_reason"].value_counts().idxmax()

print(f"Q3 answer: Lý do trả hàng phổ biến nhất của Streetwear = {q3_most_returned_reason}")

Q3 answer: Lý do trả hàng phổ biến nhất của Streetwear = wrong_size


Question 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [5]:
web_traffic = csv_io.read_csv_file(RAW_DIR / "web_traffic.csv")

q4_source_bounce = (
    web_traffic.groupby("traffic_source", as_index=False)["bounce_rate"]
    .mean()
    .sort_values("bounce_rate", ascending=True)
)

q4_best_source = q4_source_bounce.iloc[0]["traffic_source"]
q4_lowest_bounce = float(q4_source_bounce.iloc[0]["bounce_rate"])

print(f"Q4 answer: lowest-average bounce_rate source = {q4_best_source} (~ {q4_lowest_bounce:.6f})")
q4_source_bounce


Q4 answer: lowest-average bounce_rate source = email_campaign (~ 0.004458)


,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511


Question 5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?


In [6]:
order_items = csv_io.read_csv_file(RAW_DIR / "order_items.csv").copy()

order_items["promo_id"] = order_items["promo_id"].replace("", np.nan)

promo_applied_pct = order_items["promo_id"].notna().mean() * 100
print(f"Q5 answer: percentage of rows with promo_id applied ~ {promo_applied_pct:.2f}%")

Q5 answer: percentage of rows with promo_id applied ~ 38.66%


Question 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)


In [7]:
customers = csv_io.read_csv_file(RAW_DIR / "customers.csv").copy()
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv").copy()

customers["age_group"] = customers["age_group"].replace("", np.nan)

customers_byAge = customers.groupby('age_group').size().reset_index(name='total_orders')

customers_byAge

,age_group,total_orders
0,18-24,17039
1,25-34,36342
2,35-44,31920
3,45-54,23172
4,55+,13457


Question 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất (suy ra từ line_net = quantity * unit_price - discount_amount)?

In [8]:
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv").copy()
order_items = csv_io.read_csv_file(RAW_DIR / "order_items.csv").copy()
geography = csv_io.read_csv_file(RAW_DIR / "geography.csv").copy()

order_items["line_net"] = order_items["quantity"] * order_items["unit_price"] - order_items["discount_amount"]

q7_region_revenue = (
    order_items
    .merge(orders[["order_id", "zip"]], on="order_id", how="left")
    .merge(geography[["zip", "region"]], on="zip", how="left")
    .groupby("region", as_index=False)["line_net"]
    .sum()
    .sort_values("line_net", ascending=False)
)

q7_top_region = q7_region_revenue.iloc[0]["region"]
q7_option_map = {"West": "A", "Central": "B", "East": "C"}
q7_option = q7_option_map.get(q7_top_region, "D")

print(f"Q7 answer: {q7_option}) {q7_top_region}")
q7_region_revenue


Q7 answer: C) East


,region,line_net
1,East,7.291151e+09
0,Central,4.719491e+09
2,West,3.670227e+09


Question 8: Trong các đơn hàng cancelled, phương thức thanh toán nào được sử dụng nhiều nhất?

In [9]:
orders = csv_io.read_csv_file(RAW_DIR / "orders.csv").copy()

q8_payment_counts = (
    orders.loc[orders["order_status"] == "cancelled"]
    .groupby("payment_method", as_index=False)
    .size()
    .sort_values("size", ascending=False)
)

q8_top_method = q8_payment_counts.iloc[0]["payment_method"]
q8_option_map = {
    "credit_card": "A",
    "cod": "B",
    "paypal": "C",
    "bank_transfer": "D",
}
q8_option = q8_option_map.get(q8_top_method, "N/A")

print(f"Q8 answer: {q8_option}) {q8_top_method}")
q8_payment_counts


Q8 answer: A) credit_card


,payment_method,size
3,credit_card,28452
2,cod,15468
4,paypal,7817
0,apple_pay,5190
1,bank_transfer,2535


Question 9: Trong S, M, L, XL, size nào có tỷ lệ trả hàng cao nhất = số bản ghi returns / số dòng order_items (join products theo product_id)?

In [10]:
order_items = csv_io.read_csv_file(RAW_DIR / "order_items.csv").copy()
products = csv_io.read_csv_file(RAW_DIR / "products.csv").copy()
returns = csv_io.read_csv_file(RAW_DIR / "returns.csv").copy()

oi_size = order_items.merge(products[["product_id", "size"]], on="product_id", how="left", suffixes=("", "_prod"))
ret_size = returns.merge(products[["product_id", "size"]], on="product_id", how="left", suffixes=("", "_prod"))

oi_size_col = "size_prod" if "size_prod" in oi_size.columns else "size"
ret_size_col = "size_prod" if "size_prod" in ret_size.columns else "size"

q9_denominator = oi_size.groupby(oi_size_col).size().rename("order_item_rows").reset_index().rename(columns={oi_size_col: "size"})
q9_numerator = ret_size.groupby(ret_size_col).size().rename("return_records").reset_index().rename(columns={ret_size_col: "size"})

q9_size_stats = q9_denominator.merge(q9_numerator, on="size", how="left").fillna({"return_records": 0})
q9_size_stats = q9_size_stats[q9_size_stats["size"].isin(["S", "M", "L", "XL"])].copy()
q9_size_stats["return_rate"] = q9_size_stats["return_records"] / q9_size_stats["order_item_rows"]
q9_size_stats = q9_size_stats.sort_values("return_rate", ascending=False)

q9_top_size = q9_size_stats.iloc[0]["size"]
q9_option_map = {"S": "A", "M": "B", "L": "C", "XL": "D"}
q9_option = q9_option_map.get(q9_top_size, "N/A")

print(f"Q9 answer: {q9_option}) {q9_top_size}")
q9_size_stats


Q9 answer: A) S


,size,order_item_rows,return_records,return_rate
2,S,172042,9723,0.056515
0,L,173174,9741,0.056250
1,M,176428,9820,0.055660
3,XL,193025,10655,0.055200


Question 10: Kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [11]:
payments = csv_io.read_csv_file(RAW_DIR / "payments.csv").copy()

q10_per_order = payments.groupby(["installments", "order_id"], as_index=False)["payment_value"].sum()
q10_installment_avg = (
    q10_per_order
    .groupby("installments", as_index=False)["payment_value"]
    .mean()
    .sort_values("payment_value", ascending=False)
)

q10_choices = q10_installment_avg[q10_installment_avg["installments"].isin([1, 3, 6, 12])].copy()
q10_best_installment = int(q10_choices.iloc[0]["installments"])
q10_option_map = {1: "A", 3: "B", 6: "C", 12: "D"}
q10_option = q10_option_map.get(q10_best_installment, "N/A")

print(f"Q10 answer: {q10_option}) {q10_best_installment} kỳ")
q10_choices


Q10 answer: C) 6 kỳ


,installments,payment_value
3,6,24446.654403
2,3,24399.635486
4,12,24245.772694
0,1,24113.274166
